# K-Nearest Neighbors (KNN) Algorithm

*Pure Python implementation from scratch — no external ML libraries.*

---

## What is KNN?

**K-Nearest Neighbors** is one of the simplest and most intuitive machine learning algorithms. It belongs to the family of **instance-based** (or **lazy**) learning methods:

- **Lazy learning**: KNN does *no training* at all. It stores the entire training set and defers computation until prediction time. This is in contrast to **eager learners** (e.g., linear regression, neural networks) that build a model during training.
- **Non-parametric**: KNN makes no assumptions about the underlying data distribution, making it very flexible.

### How it works

Given a new data point to classify:
1. Compute the **distance** between the new point and every point in the training set.
2. Select the **k closest** training points (neighbors).
3. **Vote**: The most common class among the k neighbors becomes the prediction (for classification). For regression, the average of the neighbors' values is used.

### Distance Metrics

| Metric | Formula | Best for |
|---|---|---|
| **Euclidean** | $d = \sqrt{\sum_{i=1}^{n}(x_i - y_i)^2}$ | Continuous features, isotropic scales |
| **Manhattan** | $d = \sum_{i=1}^{n}|x_i - y_i|$ | High-dimensional data, sparse features |
| **Minkowski** | $d = \left(\sum_{i=1}^{n}|x_i - y_i|^p\right)^{1/p}$ | Generalization of the above |

### Time Complexity

- **Training**: O(1) — literally nothing to do.
- **Prediction**: O(n × d) per query, where n = number of training samples, d = number of features. This brute-force approach is costly for large datasets (optimizations like KD-trees reduce this to O(d × log n) on average).

In [ ]:
from IPython.display import Image
from math import sqrt

Image(filename='1.knn_pic.png')

## Choosing the Hyperparameter *k*

The value of **k** is a **hyperparameter** — it is not learned from data but must be set before training.

Key considerations:
1. **k should be odd** (for binary classification) to avoid ties when voting.
2. **Small k** (e.g., k=1 or k=3): The model is very sensitive to noise and outliers. This leads to **low bias but high variance** (overfitting).
3. **Large k** (e.g., k=n): The model becomes very smooth and may "dilute" the influence of nearby, high-quality neighbors. This leads to **high bias but low variance** (underfitting).

A common strategy is to try several values of k using **cross-validation** and pick the one with the best validation accuracy.

> **Rule of thumb**: Start with $k = \sqrt{n}$ where n is the number of training samples, then tune from there.

## Step 1: Euclidean Distance

The Euclidean distance between two points $\mathbf{x}$ and $\mathbf{y}$ in n-dimensional space is:

$$d(\mathbf{x}, \mathbf{y}) = \sqrt{\sum_{i=1}^{n}(x_i - y_i)^2}$$

We exclude the last element of each row because it holds the **class label**, not a feature.

In [ ]:
# Euclidean Distance

def calculate_euclidean_distance(row1, row2):
    distance = 0.0
    for i in range(len(row1) - 1):  # exclude last column (class label)
        distance += (row1[i] - row2[i]) ** 2
    return sqrt(distance)

## Step 2: Create a Toy Dataset

Each row contains two numeric features and a class label (0 or 1).

In [ ]:
# Toy dataset: [feature_1, feature_2, class_label]
dataset = [[1.80, 1.91, 0],
           [1.85, 2.11, 0],
           [2.31, 2.88, 0],
           [3.54, -3.21, 0],
           [3.66, 3.12, 0],
           [5.52, 2.13, 1],
           [6.32, 1.46, 1],
           [7.35, 2.34, 1],
           [7.78, 3.26, 1],
           [8.43, -0.34, 1]]

Let's compute the distance from the first row to every other row in the dataset:

In [ ]:
row0 = dataset[0]

for row in dataset:
    distance = calculate_euclidean_distance(row0, row)
    print(distance)

## Step 3: Find the k Nearest Neighbors

Our approach:
1. Accept an input parameter **k** (the number of neighbors).
2. Compute distances from the test point to all training points.
3. **Sort** by distance (ascending) and select the first k entries.
4. Store results as tuples `(row, distance)` for easy sorting.

### Quick aside — sorting with `lambda`

Python's `sorted()` and `list.sort()` accept a `key` argument. A **lambda function** lets us define a one-line sorting key.

In [ ]:
# Lambda sorting demo: sort a list of tuples by the third element
multi_d_list = [('f', 1, 6),
                ('c', 3, 4),
                ('d', 4, 5),
                ('b', 2, 3),
                ('a', 5, 2),
                ('e', 6, 1)]

print(sorted(multi_d_list, key=lambda x: x[2]))

### Implementing the neighbor search

In [ ]:
def get_our_neighbors(train, test_row, num_of_neighbors):
    distances = list()
    for train_row in train:
        dist = calculate_euclidean_distance(test_row, train_row)
        distances.append((train_row, dist))
    # Sort all training rows by their distance to the test row
    distances.sort(key=lambda every_tuple: every_tuple[1])
    neighbors = list()
    for i in range(num_of_neighbors):
        neighbors.append(distances[i][0])
    return neighbors

In [ ]:
neighbors = get_our_neighbors(dataset, dataset[0], 3)

for neighbor in neighbors:
    print(neighbor)

## Step 4: Make a Prediction (Majority Vote)

For **classification**, the predicted class is whichever label appears most frequently among the k neighbors.

We use `max(set(values), key=values.count)` — this finds the element in the set with the highest count in the list.

In [ ]:
def predict_the_class(train, test_row, num_of_neighbors):
    neighbors = get_our_neighbors(train, test_row, num_of_neighbors)
    the_class_values = [row[-1] for row in neighbors]
    # Majority vote: pick the class that appears most often
    prediction = max(set(the_class_values), key=the_class_values.count)
    return prediction

prediction = predict_the_class(dataset, dataset[0], 3)

In [ ]:
print('Expected (actual class): %d' % dataset[0][-1])
print('Predicted class:         %d' % prediction)

## Step 5 (Bonus): KNN for Regression

For **regression**, instead of voting, we compute the **mean** of the neighbors' target values. This variant is sometimes called **k-Nearest Neighbors Regression**.

In [ ]:
def predict_the_class_V2(train, test_row, num_of_neighbors):
    """KNN regression: predict the mean of neighbors' target values."""
    neighbors = get_our_neighbors(train, test_row, num_of_neighbors)
    the_class_values = [row[-1] for row in neighbors]
    prediction = sum(the_class_values) / float(len(the_class_values))
    return prediction

prediction = predict_the_class_V2(dataset, dataset[0], 3)

print('Expected (actual class): %d' % dataset[0][-1])
print('Predicted class:         %d' % prediction)

---

## Summary

| Aspect | Detail |
|---|---|
| **Type** | Instance-based / lazy learning |
| **Training cost** | O(1) — just store the data |
| **Prediction cost** | O(n × d) per query (brute force) |
| **Hyperparameters** | k (number of neighbors), distance metric |
| **Strengths** | Simple, no training phase, flexible decision boundaries |
| **Weaknesses** | Slow at prediction time, sensitive to irrelevant features, curse of dimensionality |

### Next Steps
- Apply this implementation to a real dataset (see the Kaggle practice projects folder).
- Experiment with different values of k and observe the effect on accuracy.
- Try Manhattan distance instead of Euclidean and compare results.